# Logistic regression using real (unsampled) data for training
## The same process was used for SMOTE-balanced data and SMOTE-balanced synthetic data 


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score
)
from sklearn.metrics import confusion_matrix

In [ ]:
real_ = pd.read_csv(r'...../Train/SGA_train.csv')
real_df = real_.drop(columns=['A', 'B', 'C', 'D'])

In [ ]:
unsampled_test_ = pd.read_csv(r'..../Test/SGA_test.csv')
#unsampled_test = unsampled_test.drop(columns=['Unnamed: 0','Imp_Focal Intestinal Perforation '], axis=1)
unsampled_test_ = unsampled_test_.drop(columns=['A', 'B', 'C', 'D'])
#list(unsampled_test_.columns)

In [ ]:
# Split and drop label

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Assuming 'df' is your DataFrame
X_r = real_df.drop('Necrotizing Enterocolitis ', axis=1) # Features
y_r = real_df['Necrotizing Enterocolitis '] # Target variable

In [ ]:
# Assuming 'df' is your DataFrame
X_test_r = unsampled_test_.drop('Necrotizing Enterocolitis ', axis=1) # Features
y_test_r = unsampled_test_['Necrotizing Enterocolitis '] # Target variable

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score

# ---------------------------------------------------
# 0. Define features and label from df
# ---------------------------------------------------
# X_all = df.drop("label", axis=1)
# y_all = df["label"]

# ---------------------------------------------------
# 1. Train / Test split
# ---------------------------------------------------
# X_train, X_test, y_train, y_test = train_test_split(
#     X_all,
#     y_all,
#     test_size=0.2,
#     stratify=y_all,
#     random_state=42
# )

# ---------------------------------------------------
# 2. Define pipeline
# ---------------------------------------------------
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000))
])

# ---------------------------------------------------
# 3. Repeated Stratified CV on TRAIN only (Step 1)
# ---------------------------------------------------
cv_scheme = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

scores = cross_validate(
    estimator=pipeline,
    X=X_r,
    y=y_r,
    cv=cv_scheme,
    scoring={
        "auc": "roc_auc",
        "auprc": "average_precision",
        "rec": "recall",
        "prec": "precision"
    },
    return_train_score=False
)

cv_metrics = pd.DataFrame({
    "Metric": ["AUROC", "AUPRC", "Recall", "Precision"],
    "Mean": [
        scores["test_auc"].mean(),
        scores["test_auprc"].mean(),
        scores["test_rec"].mean(),
        scores["test_prec"].mean()
    ],
    "CI_lower_95": [
        np.percentile(scores["test_auc"], 2.5),
        np.percentile(scores["test_auprc"], 2.5),
        np.percentile(scores["test_rec"], 2.5),
        np.percentile(scores["test_prec"], 2.5)
    ],
    "CI_upper_95": [
        np.percentile(scores["test_auc"], 97.5),
        np.percentile(scores["test_auprc"], 97.5),
        np.percentile(scores["test_rec"], 97.5),
        np.percentile(scores["test_prec"], 97.5)
    ]
})

print("Cross-validation metrics (train only):")
display(cv_metrics)

# ---------------------------------------------------
# 4. Fit on full TRAIN and evaluate on TEST (Step 2)
# ---------------------------------------------------
pipeline.fit(X_r, y_r)

y_proba_test = pipeline.predict_proba(X_test_r)[:, 1]
y_pred_test = pipeline.predict(X_test_r)

test_metrics = pd.DataFrame({
    "Metric": ["AUROC", "AUPRC", "Recall", "Precision"],
    "Value": [
        roc_auc_score(y_test_r, y_proba_test),
        average_precision_score(y_test_r, y_proba_test),
        recall_score(y_test_r, y_pred_test),
        precision_score(y_test_r, y_pred_test)
    ]
})

print("Held-out test metrics:")

In [ ]:
cnf_matrix_trtr = confusion_matrix(y_test_r, y_pred_test)
cnf_matrix_trtr

# Bootstrap real

In [ ]:
# ============================================================
# CREATE test_ci_lr
# Bootstrap 95% CI for held-out test set
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score,
    matthews_corrcoef,
    brier_score_loss
)


def bootstrap_ci_lr(
    y_true,
    y_proba,
    y_pred,
    n_boot=1000,
    random_state=42
):

    rng = np.random.RandomState(random_state)

    # Convert to numpy to avoid indexing problems
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    y_pred = np.asarray(y_pred)

    n = len(y_true)
    indices = np.arange(n)

    boot_auroc = []
    boot_auprc = []
    boot_recall = []
    boot_precision = []
    boot_f1 = []
    boot_mcc = []
    boot_brier = []

    for _ in range(n_boot):

        sample_idx = rng.choice(
            indices,
            size=n,
            replace=True
        )

        y_b = y_true[sample_idx]
        proba_b = y_proba[sample_idx]
        pred_b = y_pred[sample_idx]

        # AUROC requires both classes
        if len(np.unique(y_b)) == 2:
            boot_auroc.append(
                roc_auc_score(y_b, proba_b)
            )

        # AUPRC requires at least one positive
        if np.sum(y_b == 1) > 0:
            boot_auprc.append(
                average_precision_score(y_b, proba_b)
            )

        boot_recall.append(
            recall_score(
                y_b,
                pred_b,
                zero_division=0
            )
        )

        boot_precision.append(
            precision_score(
                y_b,
                pred_b,
                zero_division=0
            )
        )

        boot_f1.append(
            f1_score(
                y_b,
                pred_b,
                zero_division=0
            )
        )

        boot_mcc.append(
            matthews_corrcoef(
                y_b,
                pred_b
            )
        )

        boot_brier.append(
            brier_score_loss(
                y_b,
                proba_b
            )
        )

    # --------------------------------------------------------
    # Put bootstrap results into dataframe
    # --------------------------------------------------------

    results = {
        "AUROC": np.array(boot_auroc),
        "AUPRC": np.array(boot_auprc),
        "Recall": np.array(boot_recall),
        "Precision": np.array(boot_precision),
        "F1": np.array(boot_f1),
        "MCC": np.array(boot_mcc),
        "Brier Score": np.array(boot_brier)
    }

    rows = []

    for metric, values in results.items():

        rows.append({
            "Metric": metric,
            "Mean": np.mean(values),
            "CI_lower_95": np.percentile(values, 2.5),
            "CI_upper_95": np.percentile(values, 97.5)
        })

    return pd.DataFrame(rows)


# ============================================================
# RUN BOOTSTRAP
# ============================================================

test_ci_lr = bootstrap_ci_lr(
    y_true=y_test_r,
    y_proba=y_proba_test,
    y_pred=y_pred_test,
    n_boot=1000,
    random_state=42
)


print("Held-out test metrics with 95% Bootstrap CI:")

display(test_ci_lr)

# Save probabilities

In [ ]:
np.save("LR_real_proba.npy", y_proba_test)

In [ ]:
np.save("y_test_real.npy", np.asarray(y_test_r))

#### Similarly, probabilities were saved as LR_smote_proba, LR_synthetic_proba, and LR_smote_synthetic_proba

#### These probablilties were in used file Model_Results